In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm #need to install statsmodels
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
import dataframe_image as dfi

In [2]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

In [3]:
df_assess = pd.read_csv("data/cleaned/final_assessment.csv")
df_grades = pd.read_csv("data/cleaned/grades.csv")
df_status = pd.read_csv("data/cleaned/status.csv")
df_intake = pd.read_csv("data/cleaned/intake_form.csv")

In [4]:
df_intake

,id,recruitment_source,dsc_affiliation,math_affiliation,class_standing,stats_courses_taken,stats_confidence,chebyshev_familiarity,python_skill_level
0,0,DSC 140B,major,neither,Second year,"MATH 180A, MATH 181A",4,2,5
1,1,DSC 140B,major,neither,Third year,MATH 180A,4,3,5
2,3,DSC 140B,major,neither,Third year,MATH 183,4,3,4
3,6,DSC 140B,major,neither,Second year,MATH 183,3,1,4
4,9,DSC 140B,major,major,Third year,MATH 183,4,2,4
5,11,DSC 140B,major,neither,Second year,MATH 183,3,1,4
6,12,DSC 140B,major,neither,Third year,MATH 183,4,4,4
7,15,DSC 80,major,neither,Third year (first year transfer),MATH 183,3,1,4
8,20,DSC 180,major,neither,Third year (first year transfer),MATH 183,2,2,4
9,21,DSC 180,major,neither,Fourth year,ECON 120A,4,2,4


## Merge two dataframes(grades and status) by id, create coding and handwritten binary columns for b1 vs b2 test. I got rid of section 4 for now since they will affect both b1(coefficient of coding) and b2(coefficient of handwritten).

In [5]:
df_status = df_status[df_status['completed'] == 1]

sample = df_grades.merge(df_status[["id", "section"]], on="id", how="inner")

sample['coding'] =  ((sample['section'] == 2) | (sample['section'] == 4)).astype(int) #(yes,1), (no,0)
sample['handwritten'] =  ((sample['section'] == 3) | (sample['section'] == 4)).astype(int) #(yes,1), (no,0)
#sample_filter = sample[(sample['coding'] == 0) | (sample['handwritten'] == 0)]
sample

,id,coding_score,handwritten_score,final_score,final_score_adj,section,coding,handwritten
0,0,NaN,NaN,0.737705,0.616393,1,0,0
1,1,0.928571,NaN,0.754098,0.639344,2,1,0
2,3,0.857143,1.0,0.836066,0.590164,4,1,1
3,6,NaN,1.0,0.508197,0.262295,3,0,1
4,9,0.904764,NaN,0.672131,0.511475,2,1,0
5,11,0.928571,1.0,0.918033,0.439344,4,1,1
6,12,NaN,NaN,0.098361,0.019672,1,0,0
7,15,1.000000,1.0,0.344262,0.196721,4,1,1
8,20,NaN,NaN,0.573770,0.357377,1,0,0
9,21,0.928571,NaN,0.524590,0.344262,2,1,0


## Linear model for final_score = bias(section1) + b1 * coding + b2 * handwritten + b3 * (coding*written)

In [6]:
res = smf.ols(
    formula='final_score ~ coding * handwritten',
    data=sample
).fit()

print(res.summary())

s1 = res.summary2()
df_model = s1.tables[0]
df_coef  = s1.tables[1]
dfi.export(df_coef,  "data/statistic_analysis_outputs/final_separate.png", table_conversion="chrome", dpi = 300)

                            OLS Regression Results                            
Dep. Variable:            final_score   R-squared:                       0.023
Model:                            OLS   Adj. R-squared:                 -0.041
Method:                 Least Squares   F-statistic:                    0.3621
Date:                Mon, 02 Mar 2026   Prob (F-statistic):              0.781
Time:                        16:39:59   Log-Likelihood:                 10.326
No. Observations:                  50   AIC:                            -12.65
Df Residuals:                      46   BIC:                            -5.003
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              0.6293      0

## Linear model is final score = bias(section 1) + b1 * (handwritten + coding) + b2 * (coding * written)

In [7]:
res2 = smf.ols(
    formula='final_score ~ I(coding + handwritten) + coding:handwritten',
    data=sample
).fit()

print(res2.summary())
#anova_results = anova_lm(model2, model1)
s2 = res2.summary2()
df_model2 = s2.tables[0]
df_coef2  = s2.tables[1]
dfi.export(df_coef2,  "data/statistic_analysis_outputs/final_combine.png", table_conversion="chrome", dpi = 300)

                            OLS Regression Results                            
Dep. Variable:            final_score   R-squared:                       0.011
Model:                            OLS   Adj. R-squared:                 -0.031
Method:                 Least Squares   F-statistic:                    0.2607
Date:                Mon, 02 Mar 2026   Prob (F-statistic):              0.772
Time:                        16:40:01   Log-Likelihood:                 10.018
No. Observations:                  50   AIC:                            -14.04
Df Residuals:                      47   BIC:                            -8.299
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                 

## Anova Test between two models

In [8]:
anova_results = sm.stats.anova_lm(res2, res)
print(anova_results)
dfi.export(
    anova_results,
    "data/statistic_analysis_outputs/final_anova.png",
    table_conversion="chrome",
    dpi = 300
)

   df_resid       ssr  df_diff   ss_diff         F    Pr(>F)
0      47.0  1.960964      0.0       NaN       NaN       NaN
1      46.0  1.936968      1.0  0.023995  0.569846  0.454168


## Linear model for adjusted final score = bias(section1) + b1 * coding + b2 * handwritten + b3 * (coding*written)

In [9]:
res3 = smf.ols(
    formula='final_score_adj ~ coding * handwritten',
    data=sample
).fit()

print(res3.summary())

s3 = res3.summary2()
df_model3 = s3.tables[0]
df_coef3  = s3.tables[1]
dfi.export(df_coef3,  "data/statistic_analysis_outputs/final_adj_separate.png", table_conversion="chrome", dpi = 300)


                            OLS Regression Results                            
Dep. Variable:        final_score_adj   R-squared:                       0.008
Model:                            OLS   Adj. R-squared:                 -0.056
Method:                 Least Squares   F-statistic:                    0.1288
Date:                Mon, 02 Mar 2026   Prob (F-statistic):              0.943
Time:                        16:40:03   Log-Likelihood:                 9.0528
No. Observations:                  50   AIC:                            -10.11
Df Residuals:                      46   BIC:                            -2.458
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              0.4344      0

## Linear model is adjusted final score = bias(section 1) + b1 * (handwritten + coding) + b2 * (coding * written)

In [10]:
res4 = smf.ols(
    formula='final_score_adj ~ I(coding + handwritten) + coding:handwritten',
    data=sample
).fit()

print(res4.summary())
#anova_results = anova_lm(model2, model1)
s4 = res4.summary2()
df_model4 = s4.tables[0]
df_coef4  = s4.tables[1]
dfi.export(df_coef4,  "data/statistic_analysis_outputs/final_adj_combine.png", table_conversion="chrome", dpi = 300)

                            OLS Regression Results                            
Dep. Variable:        final_score_adj   R-squared:                       0.004
Model:                            OLS   Adj. R-squared:                 -0.038
Method:                 Least Squares   F-statistic:                    0.1048
Date:                Mon, 02 Mar 2026   Prob (F-statistic):              0.901
Time:                        16:40:05   Log-Likelihood:                 8.9549
No. Observations:                  50   AIC:                            -11.91
Df Residuals:                      47   BIC:                            -6.174
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                 

## Anova Test between two models for adjusted final score

In [11]:
anova_results2 = sm.stats.anova_lm(res4, res3)
print(anova_results2)
dfi.export(
    anova_results2,
    "data/statistic_analysis_outputs/final_adj_anova.png",
    table_conversion="chrome",
    dpi = 300
)

   df_resid       ssr  df_diff   ss_diff        F    Pr(>F)
0      47.0  2.046130      0.0       NaN      NaN       NaN
1      46.0  2.038132      1.0  0.007998  0.18051  0.672916


## Including variables in intake form

In [12]:
new_sample = sample.merge(df_intake, on="id", how="inner")
sample_new = new_sample.drop(columns = ['coding_score', 'handwritten_score', 'section'])
sample_new

,id,final_score,final_score_adj,coding,handwritten,recruitment_source,dsc_affiliation,math_affiliation,class_standing,stats_courses_taken,stats_confidence,chebyshev_familiarity,python_skill_level
0,0,0.737705,0.616393,0,0,DSC 140B,major,neither,Second year,"MATH 180A, MATH 181A",4,2,5
1,1,0.754098,0.639344,1,0,DSC 140B,major,neither,Third year,MATH 180A,4,3,5
2,3,0.836066,0.590164,1,1,DSC 140B,major,neither,Third year,MATH 183,4,3,4
3,6,0.508197,0.262295,0,1,DSC 140B,major,neither,Second year,MATH 183,3,1,4
4,9,0.672131,0.511475,1,0,DSC 140B,major,major,Third year,MATH 183,4,2,4
5,11,0.918033,0.439344,1,1,DSC 140B,major,neither,Second year,MATH 183,3,1,4
6,12,0.098361,0.019672,0,0,DSC 140B,major,neither,Third year,MATH 183,4,4,4
7,15,0.344262,0.196721,1,1,DSC 80,major,neither,Third year (first year transfer),MATH 183,3,1,4
8,20,0.573770,0.357377,0,0,DSC 180,major,neither,Third year (first year transfer),MATH 183,2,2,4
9,21,0.524590,0.344262,1,0,DSC 180,major,neither,Fourth year,ECON 120A,4,2,4


In [13]:
sample_new['recruitment_source'].value_counts()
#If I remember it right, class_standing(student's year) is redundant with this feature
#since older students are likely to recruited from a uppertive class

heard_type = {
    'DSC 10': 'Heard_1',
    'DSC 20': 'Heard_1',
    'DSC 30': 'Heard_1',
    'DSC 40A': 'Heard_2',
    'DSC 40B': 'Heard_2',
    'DSC 80': 'Heard_2',
    'DSC 140B': 'Heard_3',
    'CSE 150A': 'Heard_3',
    'CSE 151A': 'Heard_3',
    'DSC 180': 'Heard_3',
    'DSC 100': 'Heard_4',
    'DSC 120': 'Heard_4',
    'DSC 190': 'Heard_4',
}

In [14]:
sample_new['dsc_affiliation'].value_counts()
dsc_type = {
    'major': 'DSC_major',
    'minor': 'DSC_minor',
    'neither': 'DSC_neither',
}

In [15]:
sample_new['math_affiliation'].value_counts()

math_type = {
    'major': 'MATH_major',
    'minor': 'MATH_minor',
    'neither': 'MATH_neither',
}

In [16]:
sample_new['class_standing'].value_counts()
year_type = {
    'First year': 'Year_1',
    'Second year': 'Year_2',
    'Third year': 'Year_3',
    'Fourth year': 'Year_4',
    'Third year (first year transfer)': 'Year_transfer_1',
    'Fourth year (second year transfer)': 'Year_transfer_2'
}

In [17]:
sample_new['stats_courses_taken'].value_counts()

section_type = {
    'MATH 181A': 'Stats_1',
    'MATH 180A': 'Stats_2',
    'ECE 109': 'Stats_2',
    'MAE 108': 'Stats_2',
    'MATH 183': 'Stats_3',
    'MATH 186': 'Stats_3',
    'ECON 120A': 'Stats_3',
    'None of the above': 'Stats_4'
}

In [18]:
def mappings(col, dictionary):
    values = [i.strip() for i in col.split(',')]
    adjust = {dictionary.get(j) for j in values if j in dictionary} #keep unique values
    return ', '.join(adjust)

In [19]:
samples = sample_new.copy()
samples['recruitment_source'] = samples['recruitment_source'].apply(lambda x: mappings(x, heard_type))
samples['stats_courses_taken'] = samples['stats_courses_taken'].apply(lambda x: mappings(x, section_type))
samples['dsc_affiliation'] = samples['dsc_affiliation'].apply(lambda x: mappings(x, dsc_type))
samples['math_affiliation'] = samples['math_affiliation'].apply(lambda x: mappings(x, math_type))
samples['class_standing'] = samples['class_standing'].apply(lambda x: mappings(x, year_type))
samples

,id,final_score,final_score_adj,coding,handwritten,recruitment_source,dsc_affiliation,math_affiliation,class_standing,stats_courses_taken,stats_confidence,chebyshev_familiarity,python_skill_level
0,0,0.737705,0.616393,0,0,Heard_3,DSC_major,MATH_neither,Year_2,"Stats_2, Stats_1",4,2,5
1,1,0.754098,0.639344,1,0,Heard_3,DSC_major,MATH_neither,Year_3,Stats_2,4,3,5
2,3,0.836066,0.590164,1,1,Heard_3,DSC_major,MATH_neither,Year_3,Stats_3,4,3,4
3,6,0.508197,0.262295,0,1,Heard_3,DSC_major,MATH_neither,Year_2,Stats_3,3,1,4
4,9,0.672131,0.511475,1,0,Heard_3,DSC_major,MATH_major,Year_3,Stats_3,4,2,4
5,11,0.918033,0.439344,1,1,Heard_3,DSC_major,MATH_neither,Year_2,Stats_3,3,1,4
6,12,0.098361,0.019672,0,0,Heard_3,DSC_major,MATH_neither,Year_3,Stats_3,4,4,4
7,15,0.344262,0.196721,1,1,Heard_2,DSC_major,MATH_neither,Year_transfer_1,Stats_3,3,1,4
8,20,0.573770,0.357377,0,0,Heard_3,DSC_major,MATH_neither,Year_transfer_1,Stats_3,2,2,4
9,21,0.524590,0.344262,1,0,Heard_3,DSC_major,MATH_neither,Year_4,Stats_3,4,2,4


In [20]:
df_encoded = pd.concat([samples, samples['stats_courses_taken'].str.get_dummies(sep=', '), 
                        samples['recruitment_source'].str.get_dummies(sep=', '),
                        samples['dsc_affiliation'].str.get_dummies(sep=', '),
                        samples['math_affiliation'].str.get_dummies(sep=', '),
                        samples['class_standing'].str.get_dummies(sep=', '),], axis = 1)
df_encoded = df_encoded.drop(columns = ['recruitment_source', 'dsc_affiliation', 'math_affiliation', 'stats_courses_taken', 'Stats_4',
                           'DSC_neither', 'MATH_neither', 'class_standing'], errors='ignore') 
#dropping all string columns and neither columns to prevent redundency

#needed for backward elimination, this is for people who are in section 4, who coding and handwritten problem sets
df_encoded['coding_handwritten'] = df_encoded['coding']*df_encoded['handwritten']
#reorder the columns
df_encoded = df_encoded[list(df_encoded.columns[:5]) + ['coding_handwritten'] + list(df_encoded.columns[5:-1])]
df_encoded

,id,final_score,final_score_adj,coding,handwritten,coding_handwritten,stats_confidence,chebyshev_familiarity,python_skill_level,Stats_1,Stats_2,Stats_3,Heard_1,Heard_2,Heard_3,Heard_4,DSC_major,DSC_minor,MATH_major,MATH_minor,Year_1,Year_2,Year_3,Year_4,Year_transfer_1,Year_transfer_2
0,0,0.737705,0.616393,0,0,0,4,2,5,1,1,0,0,0,1,0,1,0,0,0,0,1,0,0,0,0
1,1,0.754098,0.639344,1,0,0,4,3,5,0,1,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0
2,3,0.836066,0.590164,1,1,1,4,3,4,0,0,1,0,0,1,0,1,0,0,0,0,0,1,0,0,0
3,6,0.508197,0.262295,0,1,0,3,1,4,0,0,1,0,0,1,0,1,0,0,0,0,1,0,0,0,0
4,9,0.672131,0.511475,1,0,0,4,2,4,0,0,1,0,0,1,0,1,0,1,0,0,0,1,0,0,0
5,11,0.918033,0.439344,1,1,1,3,1,4,0,0,1,0,0,1,0,1,0,0,0,0,1,0,0,0,0
6,12,0.098361,0.019672,0,0,0,4,4,4,0,0,1,0,0,1,0,1,0,0,0,0,0,1,0,0,0
7,15,0.344262,0.196721,1,1,1,3,1,4,0,0,1,0,1,0,0,1,0,0,0,0,0,0,0,1,0
8,20,0.573770,0.357377,0,0,0,2,2,4,0,0,1,0,0,1,0,1,0,0,0,0,0,0,0,1,0
9,21,0.524590,0.344262,1,0,0,4,2,4,0,0,1,0,0,1,0,1,0,0,0,0,0,0,1,0,0


In [21]:
df_encoded.columns

Index(['id', 'final_score', 'final_score_adj', 'coding', 'handwritten', 'coding_handwritten', 'stats_confidence', 'chebyshev_familiarity',
       'python_skill_level', 'Stats_1', 'Stats_2', 'Stats_3', 'Heard_1', 'Heard_2', 'Heard_3', 'Heard_4', 'DSC_major', 'DSC_minor', 'MATH_major', 'MATH_minor',
       'Year_1', 'Year_2', 'Year_3', 'Year_4', 'Year_transfer_1', 'Year_transfer_2'],
      dtype='object')

## Moving on with backward elimnination, we don't want our model to overfit.

## Here are all of the input features we will use to calculate the final_score and final_score_adj

In [22]:
initial_vars = [col for col in df_encoded.columns if col != "id" and col != 'final_score' and col != 'final_score_adj']
initial_vars

['coding',
 'handwritten',
 'coding_handwritten',
 'stats_confidence',
 'chebyshev_familiarity',
 'python_skill_level',
 'Stats_1',
 'Stats_2',
 'Stats_3',
 'Heard_1',
 'Heard_2',
 'Heard_3',
 'Heard_4',
 'DSC_major',
 'DSC_minor',
 'MATH_major',
 'MATH_minor',
 'Year_1',
 'Year_2',
 'Year_3',
 'Year_4',
 'Year_transfer_1',
 'Year_transfer_2']

## Reorganizing the feature:
## First the features we need to include: forced_vars   list of strings
## Second the features that we will consider to eliminate: all_inputs list of strings and lists, the reason it contains list is that it contains one hot encoded feature, when we remove it, we want to remove all those kind of feature at once.

In [23]:
forced_vars = ['coding', 'handwritten', 'coding_handwritten']

In [24]:
add_on = list(set(initial_vars) - set(forced_vars))
add_on

['chebyshev_familiarity',
 'Heard_1',
 'Year_3',
 'Heard_2',
 'Year_transfer_1',
 'Year_transfer_2',
 'stats_confidence',
 'Heard_3',
 'Stats_1',
 'Year_1',
 'DSC_major',
 'python_skill_level',
 'Year_4',
 'DSC_minor',
 'MATH_minor',
 'Year_2',
 'MATH_major',
 'Stats_2',
 'Heard_4',
 'Stats_3']

## Features being one hot encoded

In [25]:
same_feature = ['Stats', 'Heard', 'DSC', 'MATH', 'Year']
stats = []
heard = []
dsc = []
math = []
year = []
all_inputs = []
for i in add_on:
    if i.startswith("Stats"):
        stats.append(i)
    elif i.startswith("Heard"):
        heard.append(i)
    elif i.startswith("DSC"):
        dsc.append(i)
    elif i.startswith("MATH"):
        math.append(i)
    elif i.startswith("Year"):
        year.append(i)
    else:
        all_inputs.append(i)

all_inputs.append(stats)
all_inputs.append(heard)
all_inputs.append(dsc)
all_inputs.append(math)
all_inputs.append(year)        
all_inputs

['chebyshev_familiarity',
 'stats_confidence',
 'python_skill_level',
 ['Stats_1', 'Stats_2', 'Stats_3'],
 ['Heard_1', 'Heard_2', 'Heard_3', 'Heard_4'],
 ['DSC_major', 'DSC_minor'],
 ['MATH_minor', 'MATH_major'],
 ['Year_3',
  'Year_transfer_1',
  'Year_transfer_2',
  'Year_1',
  'Year_4',
  'Year_2']]

In [26]:
def backward_selection_adjR2(data, all_inputs, response, forced_vars):
    """
    Backward elimination based on Adjusted R^2.
    Forced variables are always included.
    Groups in all_inputs are removed together.
    """
        
    # ---- Step 1: build removable groups ----

    
    groups = []
    for item in all_inputs:
        if isinstance(item, list):
            groups.append(item)
        else:
            groups.append([item])
    
    # taking forced_vars away from removable groups
    removable_groups = []
    for group in groups:
        group_filtered = [v for v in group if v not in forced_vars]
        if group_filtered:
            removable_groups.append(group_filtered)

    #used to generate full model to compare with each one variable eliniated model
    def flatten(group_list):
        return [v for group in group_list for v in group]
    
    count = 0
    
    while True:
        
        # all features = forced_vars + removable features
        current_vars = list(set(forced_vars + flatten(removable_groups)))
        
        formula = response + " ~ " + " + ".join(current_vars)
        model = smf.ols(formula=formula, data=data).fit()
        current_adjR2 = model.rsquared_adj
        
        #current full model adj R^2 and create a variable to find the variable we want to remove for this iteration
        best_adjR2 = current_adjR2
        best_group_to_remove = None
        
        # taking one feature away and compare with the full model
        for group in removable_groups:
            
            temp_groups = removable_groups.copy()
            temp_groups.remove(group)
            
            temp_vars = list(set(forced_vars + flatten(temp_groups)))
            
            formula = response + " ~ " + " + ".join(temp_vars)
            temp_model = smf.ols(formula=formula, data=data).fit()
            
            #if the adjusted R^2 increased, we then keep the record of this inputs
            if temp_model.rsquared_adj > best_adjR2:
                best_adjR2 = temp_model.rsquared_adj
                best_group_to_remove = group

        #stop when adjusted R^2 no longer increase
        if best_group_to_remove is None:
            break
        
        #after the inner for loop, this will remove the feature that increase adjusted R^2 the most
        removable_groups.remove(best_group_to_remove)
        count += 1
        
        print(f"\nIteration {count}")
        print(f"Removed group: {best_group_to_remove}")
        print(f"New Adjusted R^2: {best_adjR2:.6f}")
        print("="*80)
    
    # ---- Final model ----
    
    final_vars = list(set(forced_vars + flatten(removable_groups)))
    final_formula = response + " ~ " + " + ".join(final_vars)
    final_model = smf.ols(formula=final_formula, data=data).fit()
    
    print(f"\nBackward elimination ran {count} iterations.")
    print("\nFinal formula:")
    print(final_formula)
    
    return final_model.summary()

## This is the backward elimination with final_score as the predict variable with final model at the end.

In [27]:
backward_selection_adjR2(df_encoded, all_inputs, 'final_score', forced_vars)


Iteration 1
Removed group: ['Heard_1', 'Heard_2', 'Heard_3', 'Heard_4']
New Adjusted R^2: 0.001955

Iteration 2
Removed group: ['stats_confidence']
New Adjusted R^2: 0.029787

Iteration 3
Removed group: ['chebyshev_familiarity']
New Adjusted R^2: 0.056809

Iteration 4
Removed group: ['python_skill_level']
New Adjusted R^2: 0.079530

Iteration 5
Removed group: ['MATH_minor', 'MATH_major']
New Adjusted R^2: 0.084381

Backward elimination ran 5 iterations.

Final formula:
final_score ~ coding_handwritten + Year_3 + Year_4 + DSC_minor + handwritten + Year_1 + Year_2 + Year_transfer_1 + Year_transfer_2 + Stats_2 + coding + Stats_3 + Stats_1 + DSC_major


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:            final_score   R-squared:                       0.327
Model:                            OLS   Adj. R-squared:                  0.084
Method:                 Least Squares   F-statistic:                     1.347
Date:                Mon, 02 Mar 2026   Prob (F-statistic):              0.233
Time:                        16:40:08   Log-Likelihood:                 19.653
No. Observations:                  50   AIC:                            -11.31
Df Residuals:                      36   BIC:                             15.46
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
======================================================================================
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              0.5560      0.085      6.527      0.000       0.383       0.729
coding_handwritten    -0.0226      0.120     -0.189      0.852      -0.266       0.221
Year_3                 0.0962      0.073      1.320      0.195      -0.052       0.244
Year_4                 0.0901      0.073      1.232      0.226      -0.058       0.238
DSC_minor             -0.1647      0.089     -1.860      0.071      -0.344       0.015
handwritten            0.0949      0.085      1.122      0.269      -0.077       0.267
Year_1                 0.1203      0.063      1.907      0.064      -0.008       0.248
Year_2                 0.1019      0.084      1.219      0.231      -0.068       0.271
Year_transfer_1       -0.0895      0.078     -1.143      0.261      -0.248       0.069
Year_transfer_2        0.2369      0.108      2.192      0.035       0.018       0.456
Stats_2                0.1919      0.094      2.031      0.050       0.000       0.383
coding                 0.0563      0.085      0.663      0.511      -0.116       0.229
Stats_3                0.0787      0.084      0.936      0.356      -0.092       0.249
Stats_1                0.0448      0.122      0.366      0.717      -0.203       0.293
DSC_major             -0.1673      0.085     -1.976      0.056      -0.339       0.004
==============================================================================
Omnibus:                        1.732   Durbin-Watson:                   2.084
Prob(Omnibus):                  0.421   Jarque-Bera (JB):                1.411
Skew:                          -0.410   Prob(JB):                        0.494
Kurtosis:                       2.922   Cond. No.                     1.57e+16
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The smallest eigenvalue is 4.75e-31. This might indicate that there are
strong multicollinearity problems or that the design matrix is singular.
"""

## This is the backward elimination with final_score_adj as the predict variable with final model at the end.

In [28]:
backward_selection_adjR2(df_encoded, all_inputs, 'final_score_adj', forced_vars)


Iteration 1
Removed group: ['Heard_1', 'Heard_2', 'Heard_3', 'Heard_4']
New Adjusted R^2: 0.168767

Iteration 2
Removed group: ['MATH_minor', 'MATH_major']
New Adjusted R^2: 0.203840

Iteration 3
Removed group: ['python_skill_level']
New Adjusted R^2: 0.222685

Iteration 4
Removed group: ['stats_confidence']
New Adjusted R^2: 0.231890

Iteration 5
Removed group: ['chebyshev_familiarity']
New Adjusted R^2: 0.242632

Backward elimination ran 5 iterations.

Final formula:
final_score_adj ~ coding_handwritten + Year_3 + Year_4 + DSC_minor + handwritten + Year_1 + Year_2 + Year_transfer_1 + Year_transfer_2 + Stats_2 + coding + Stats_3 + Stats_1 + DSC_major


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:        final_score_adj   R-squared:                       0.444
Model:                            OLS   Adj. R-squared:                  0.243
Method:                 Least Squares   F-statistic:                     2.208
Date:                Mon, 02 Mar 2026   Prob (F-statistic):             0.0305
Time:                        16:40:08   Log-Likelihood:                 23.499
No. Observations:                  50   AIC:                            -19.00
Df Residuals:                      36   BIC:                             7.771
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
======================================================================================
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              0.4250      0.079      5.388      0.000       0.265       0.585
coding_handwritten     0.0410      0.111      0.369      0.714      -0.184       0.266
Year_3                 0.1199      0.068      1.775      0.084      -0.017       0.257
Year_4                 0.0991      0.068      1.464      0.152      -0.038       0.236
DSC_minor             -0.2198      0.082     -2.680      0.011      -0.386      -0.053
handwritten            0.0567      0.078      0.724      0.474      -0.102       0.216
Year_1                 0.0685      0.058      1.172      0.249      -0.050       0.187
Year_2                 0.0050      0.077      0.064      0.949      -0.152       0.162
Year_transfer_1       -0.1016      0.073     -1.400      0.170      -0.249       0.046
Year_transfer_2        0.2341      0.100      2.339      0.025       0.031       0.437
Stats_2                0.1628      0.087      1.861      0.071      -0.015       0.340
coding                 0.0106      0.079      0.135      0.893      -0.149       0.170
Stats_3                0.0306      0.078      0.393      0.697      -0.127       0.188
Stats_1                0.1234      0.113      1.089      0.284      -0.107       0.353
DSC_major             -0.1715      0.078     -2.188      0.035      -0.331      -0.013
==============================================================================
Omnibus:                        0.407   Durbin-Watson:                   2.045
Prob(Omnibus):                  0.816   Jarque-Bera (JB):                0.210
Skew:                          -0.159   Prob(JB):                        0.900
Kurtosis:                       2.978   Cond. No.                     1.57e+16
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The smallest eigenvalue is 4.75e-31. This might indicate that there are
strong multicollinearity problems or that the design matrix is singular.
"""